# 9. AIND Ephys QC Collector

Run [aind-ephys-qc-collector](https://github.com/AllenNeuralDynamics/aind-ephys-qc-collector/tree/main/code) on the per-recording QC produced by `08_aind_ephys_processing_qc.ipynb`.

The collector reads all `quality_control_<name>.json` files (and matching `quality_control_<name>/` figure folders), rewrites figure references, and writes a single aggregated `quality_control.json` document plus a flat `quality_control/<probe>/` figure tree into `../results/`.

## Imports and deps

In [1]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

In [2]:
subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "aind-data-schema"],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv
Resolved 14 packages in 10ms
Uninstalled 2 packages in 6ms
Installed 2 packages in 2ms
 - pydantic==2.12.5
 + pydantic==2.11.10
 - pydantic-core==2.41.5
 + pydantic-core==2.33.2


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'aind-data-schema'], returncode=0)

## Clone the capsule and seed `data/`

Just the per-recording QC outputs from notebook 8.

In [3]:
qcc_repo = Path("/tmp/aind-ephys-qc-collector")
if not qcc_repo.exists():
    subprocess.run(
        [
            "git", "clone", "--depth=1",
            "https://github.com/AllenNeuralDynamics/aind-ephys-qc-collector.git",
            str(qcc_repo),
        ],
        check=True,
    )

data_dir = qcc_repo / "data"
results_dir = qcc_repo / "results"
data_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

for stale in list(data_dir.iterdir()) + list(results_dir.iterdir()):
    shutil.rmtree(stale) if stale.is_dir() else stale.unlink()

src = Path.cwd() / "output/08_qc_results"
assert src.exists(), "Run 08_aind_ephys_processing_qc.ipynb first."
for entry in src.iterdir():
    dest = data_dir / entry.name
    shutil.copytree(entry, dest) if entry.is_dir() else shutil.copy2(entry, dest)

print("Seeded data dir:")
for p in sorted(data_dir.iterdir()):
    print(" ", p.name)

Cloning into '/tmp/aind-ephys-qc-collector'...


Seeded data dir:
  quality_control_block0_None_recording1
  quality_control_block0_None_recording1.json


## Run the QC collector

No CLI args, no params.json — it discovers everything in `data/`.

In [4]:
argv = ["python", "-u", "run_capsule.py"]
print("Running:", " ".join(argv))
result = subprocess.run(
    argv,
    cwd=qcc_repo / "code",
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:\n", result.stderr)
    raise RuntimeError(f"qc-collector run failed with code {result.returncode}")

Running: python -u run_capsule.py



EPHYS QC COLLECTION
Found 1 quality control files
All recording segments are the same. Dropping recording segment from recording name.
	Collected 5 metrics for 0 tags and 1 streams.
	Tags allowed to fail: []
EPHYS QC COLLECTION time: 0.0s



## Copy outputs next to the notebook

In [5]:
local_results_dir = Path.cwd() / "output/09_qc_collected_results"
local_results_dir.parent.mkdir(parents=True, exist_ok=True)
if local_results_dir.exists():
    shutil.rmtree(local_results_dir)
shutil.copytree(results_dir, local_results_dir)

for p in sorted(local_results_dir.rglob("*")):
    rel = p.relative_to(local_results_dir)
    kind = "dir" if p.is_dir() else f"{p.stat().st_size} bytes"
    print(f"  {rel}  ({kind})")

  quality_control  (dir)
  quality_control/block0_None  (dir)
  quality_control/block0_None/firing_rate.png  (80291 bytes)
  quality_control/block0_None/psd.png  (309323 bytes)
  quality_control/block0_None/rms.png  (135541 bytes)
  quality_control/block0_None/traces_raw.png  (415823 bytes)
  quality_control/block0_None/unit_yield.png  (333995 bytes)
  quality_control.json  (6368 bytes)


## Inspect the aggregated quality_control.json

In [6]:
qc = json.loads((local_results_dir / "quality_control.json").read_text())
print("schema_version:", qc.get("schema_version"))
print("default_grouping:", qc.get("default_grouping"))
print("allow_tag_failures:", qc.get("allow_tag_failures"))
print(f"Total metrics: {len(qc.get('metrics', []))}")
for m in qc.get("metrics", []):
    print(f"  {m.get('stage'):>10}  {m.get('name'):40}  reference={m.get('reference')}")

schema_version: 2.4.1
default_grouping: ['probe', 'stage']
allow_tag_failures: []
Total metrics: 5
    Raw data  Raw data block0_None                      reference=quality_control/block0_None/traces_raw.png
    Raw data  PSD block0_None                           reference=quality_control/block0_None/psd.png
    Raw data  RMS block0_None                           reference=quality_control/block0_None/rms.png
  Processing  Unit Metrics Yield - block0_None          reference=quality_control/block0_None/unit_yield.png
  Processing  Firing rate - block0_None                 reference=quality_control/block0_None/firing_rate.png
